# Compliance evidence: time error, UTC traceability and an audit report

Regulated users (MiFID II RTS 25, DORA, NIS2) and telecom operators (ITU-T G.8271/G.8273) need
numbers they can defend. This notebook produces three:

1. **time error** of a PTP client from a packet capture, checked against limits;
2. an **error bound to UTC** for an NTP client, with its assumptions;
3. a self-contained **HTML report** with input hashes, to archive.

<sub>MIT licensed, part of [ntpstats](https://github.com/thiagodefreitas/NetworkTime). Runs in CI on every change.</sub>

In [ ]:
from pathlib import Path

from ntpstats import api as nt

# examples/data, whether the notebook runs from examples/notebooks or the repository root
DATA = next(p for p in (Path("../data"), Path("examples/data"), Path("../../examples/data")) if p.is_dir())

## PTP time error from a capture

The capture is analysed from the capture host's clock: each Sync/Delay exchange gives an offset.
`time_error` computes max|TE|, constant TE (cTE) and dynamic TE (dTE, low- and high-pass) and
`check_time_error` compares them with limits (here G.8273.2 class B-like values).

In [ ]:
ptp = nt.load_one(DATA / "ptp-capture.pcapng")
te = nt.time_error(ptp, cte_window=60)
limits = {"max_te": 100e-9, "cte": 50e-9, "dte_l_pp": 70e-9}
verdict = nt.check_time_error(te, limits)
print(f"{len(ptp)} exchanges from {ptp.name}")
for k, v in te.metrics().items():
    print(f"{k:>10}: {nt.format_seconds(v)}")
for row in verdict["checks"]:
    print(f"{row['label']:>22} {nt.format_seconds(row['value']):>10} <= {nt.format_seconds(row['limit']):>10}:",
          "PASS" if row["passed"] else "FAIL")
print("overall:", "PASS" if verdict["passed"] else "FAIL")

## NTP client: error bound to UTC

`audit` turns the measured offsets, the round-trip delay (worst-case asymmetry is half of it),
the server's root distance and the stated uncertainty of the reference into a per-sample bound,
then reports the share of time within the limit. RTS 25 asks 100 µs or 1 ms depending on
the activity.

In [ ]:
s = nt.load_one(DATA / "chrony-tracking.log")
events = nt.detect_events(s)
result = nt.audit(s, nt.AuditConfig(limit=1e-3), events=events)
print(f"{len(events)} events detected")
for e in events[:5]:
    print(f"   {e.kind} at {e.time:.0f}: {e.magnitude:.3g} {e.unit}")
for k, v in result["summary"].items():
    print(f"{k:>24}: {v}")
for note in result["notes"][:3]:
    print("note:", note)

This client **fails** a 1 ms limit, and the audit says why: its server is 11 ms away, and without
a measurement of the path asymmetry, half the round-trip delay must be counted as possible error.
Proving 1 ms needs a closer (or PTP/PPS) reference, or a measured asymmetry
(`AuditConfig(asymmetry=...)`). The bound is honest rather than optimistic, which is what an
auditor needs.

## Stability against a mask

A mask is a CSV of `tau,limit` pairs; `check_mask` gives the margin at each τ.

In [ ]:
import tempfile

with tempfile.NamedTemporaryFile("w", suffix=".csv", delete=False) as fh:
    fh.write("tau,tdev\n64,5e-3\n8192,1e-2\n")
mask = nt.load_mask(fh.name)
tdev = nt.series_stability(s, kinds=("tdev",))[0]
res = nt.check_mask(tdev, mask)
print("PASS" if res["passed"] else "FAIL", f"(worst margin: {res['worst_margin']:.2f} x the measured TDEV)")

## An archivable report

`dataset_report` writes one offline HTML file with the charts, the tables, the parameters and
SHA-256 hashes of the inputs.

In [ ]:
html = nt.dataset_report([s], kinds=("oadev", "tdev"), inputs=[str(DATA / "chrony-tracking.log")],
                         title="NTP client evidence", time_error_section=True)
out = Path(tempfile.gettempdir()) / "ntp-evidence.html"
out.write_text(html, encoding="utf-8")
print(f"{out} ({len(html) // 1024} kB)")